In [ ]:
pip install findspark

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [ ]:
import os
import findspark
#from pyspark import SparkConf, SparkContext
#from pyspark.sql import SparkSession
#import pyspark
import time
#from pyspark.sql import functions as sf

# --- Set the SPARK_HOME environment variable manually ---
os.environ["SPARK_HOME"] = "C:/spark" 
findspark.init()
print("Findspark initialized successfully.") 

In [ ]:
from pyspark.sql import SparkSession
import time
import os
from pyspark import SparkConf, SparkContext
import pyspark
from pyspark.sql import functions as sf

# --- Configuration ---
# Ensure the path uses forward slashes or raw strings for Python compatibility
spark_connector_path = r"C:\spark_drivers\spark-mssql-connector_2.12-1.1.0.jar"
#mssql_jdbc_path = r"C:\spark_drivers\mssql-jdbc-11.2.4.jre11.jar" # <-- Assuming you have the standard JDBC driver as well
#mssql-jdbc-8.4.1.jre8.jar
mssql_jdbc_path = r"C:\spark_drivers\mssql-jdbc-8.4.1.jre8.jar"
# Combine all local JARs paths into a comma-separated string
all_jars_paths = f"{spark_connector_path},{mssql_jdbc_path}"

sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=AdventureSales;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
source_view_name = "sttwk1b"
sqlsvr_destination_table = "STTWK_2225"
# ---------------------

# Create Spark Session using the LOCAL JAR files via spark.jars config
# You MUST restart your Python kernel before running this cell if Spark was already active
# spark 3.2 and below
# Add the necessary JVM flags to fix the IllegalAccessError on newer JDKs (11+)
jvm_options = "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED"

# Create Spark Session using the LOCAL JAR files via spark.jars config
# You MUST restart your Python kernel before running this cell if Spark was already active
spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer_LocalJars") \
    .config("spark.jars", all_jars_paths) \
    .config("spark.executor.extraJavaOptions", jvm_options) \
    .config("spark.driver.extraJavaOptions", jvm_options) \
    .getOrCreate()
    
print(f"The active Spark version is: {spark.version}")
print("Spark session created with local JARs configured.")

# spark 3.3
'''
spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer_LocalJars") \
    .config("spark.jars", all_jars_paths) \
    .getOrCreate()
'''    
print(f"The active Spark version is: {spark.version}")
print("Spark session created with local JARs configured.")

# ... rest of your code ...

start_time = time.time()
df_partitioned = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", source_view_name) \
    .option("fetchsize", 1000) \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("driver", sqlserver_properties["driver"]) \
    .load()
end_time = time.time()
print(f"Total time taken for load: {end_time - start_time:.2f} seconds")

print(f"Current Spark Version: {spark.version}")
conf = spark.sparkContext.getConf()
print("--- Key Spark Configurations ---")
print(f"App Name: {conf.get('spark.app.name')}")
print(f"Spark Version: {spark.version}")

# This configuration entry shows which JARs are loaded locally
loaded_jars = conf.get('spark.jars')
print(f"Loaded JAR paths: {loaded_jars}")

# This configuration entry shows which packages Spark tried to download from Maven
loaded_packages = conf.get('spark.jars.packages')
print(f"Loaded Maven packages: {loaded_packages}")

# Your write operation should now work as all necessary drivers are loaded
start_time = time.time()

df_partitioned.write \
    .format("com.microsoft.sqlserver.jdbc.spark") \
    .option("url", sql_server_url ) \
    .option("dbtable", sqlsvr_destination_table) \
    .option("batchsize", 100000) \
    .mode("overwrite") \
    .save()
'''
df_partitioned.write \
    .format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", sqlsvr_destination_table) \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("driver", sqlserver_properties["driver"]) \
    .option("batchsize", 100000) \
    .mode("overwrite") \
    .save() # Note: Default mode is ErrorIfExists. You might want .mode("append") or .mode("overwrite")
'''    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for write: {total_time:.2f} seconds")



In [ ]:
start_time = time.time()
df_partitioned.write \
    .format("com.microsoft.sqlserver.jdbc.spark") \
    .option("url", sql_server_url ) \
    .option("dbtable", sqlsvr_destination_table) \
    .option("batchsize", 100000) \
    .save()
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for write: {total_time:.2f} seconds")

In [ ]:
from pyspark.sql.functions import countDistinct, col

# Start the timer
start_time = time.time()

# Option A: Repartition by the character column (hash partitioning)
# The data associated with each unique value in 'char_column' will go to a single partition.
#df_to_write = sql_server_df.repartition(181, "year_week_int")
num_read_partitions = df_partitioned.agg(countDistinct(col("year_week_int"))).collect()[0][0]
print(f"Detected {num_read_partitions} unique year_week_int values.")
df_to_write = df_partitioned.repartition(num_read_partitions, "year_week_int")
# Option B: Repartition generally (even hash distribution)
# Use this if you don't care about co-locating data by the char column, 
# or if the data in 'your_char_column_name' is very skewed.
# df_to_write = sql_server_df.repartition(10) 

print(f"DataFrame has {df_to_write.rdd.getNumPartitions()} partitions before write.")
predicates = [f"{partition_column_name} = '{yw_value}'" for yw_value in year_week_list]
#columns_to_select = "branch_id, sp_id, custd_id, prod_id, sls_id, yr, mth, wk, net_qty, net_amt, grs_qty, grs_amt, rtn_qty, rtn_amt"

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

In [ ]:
# Drop the temporary 'year_week_int' column before writing to Oracle
# just do this cell if dont want to include them !!!!
#df_to_write = df_partitioned.drop("year_week_int")


In [ ]:
# Assuming you have successfully run the previous code block which creates `df_partitioned`

import time

# --- Configuration for Destination Table ---
sqlsvr_destination_table = "sttwk_2225" 

# Define write properties with bulkCopy AND truncate enabled
write_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "encrypt": "true",
    "trustServerCertificate": "true",
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}

print(f"Attempting to truncate and write to destination table: {sqlsvr_destination_table}")
#print(f"DataFrame has {df_partitioned.rdd.getNumPartitions()} partitions for writing.")

# Start timer for write operation
write_start_time = time.time()

try:
    df_to_write.write \
        .mode("overwrite") \
        .jdbc(
            url=sql_server_url, 
            table=sqlsvr_destination_table, 
            properties=write_properties
        )
    
    write_end_time = time.time()
    write_total_time = write_end_time - write_start_time

    print("-" * 50)
    print(f"Data successfully truncated and inserted into table {sqlsvr_destination_table}.")
    print(f"Write operation completed in: {write_total_time:.2f} seconds")
    print("-" * 50)

except Exception as e:
    print(f"An error occurred during the write operation: {e}")   

In [ ]:
# Paste the following modified code into your active PySpark shell session
import time
# --- Configuration (UPDATE THESE VALUES) ---
# Start the timer
start_time = time.time()
JDBC_URL = "jdbc:sqlserver://HYPE\\SQL_2022:1433;"
SOURCE_DB_NAME = "AdventureSales" 
#TARGET_TABLE = "Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint"
TARGET_TABLE = "sttwk_2225"
USERNAME = "sa"
PASSWORD = "D@1ry$2014"
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" 

# Connection Properties for the READ operation (no change needed here)
read_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
}
# Connection Properties for the WRITE operation
# We rely on the default OVERWRITE behavior (DROP and CREATE TABLE)
write_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
    # Remove "truncate": "true" here
    # --- ADD THIS LINE FOR BULK INSERT SPEED ---
    "bulkCopy": "true",
    # --- KEY CHANGE: Truncate instead of Drop ---
    "truncate": "true" 
}


# --- 1. Define the Source SQL Query (No change) ---
source_sql_query = """
(SELECT 
    *
FROM sttwk1b 
) as custom_query_alias
"""

# --- 2. Load data from SQL Server into a PySpark DataFrame ---
source_df = spark.read.jdbc(
    url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
    table=source_sql_query,
    properties=read_properties
)

print(f"Loaded {source_df.count()} records from source SQL query into PySpark DataFrame.")
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")  

# --- 3. Overwrite the Target Table (DROP and CREATE TABLE approach) ---
# Start the timer
start_time = time.time()
optimized_df = source_df.repartition(52)
try:
    # Use the optimized DataFrame
    optimized_df.write \
        .mode("overwrite") \
        .jdbc(
            url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
            table=TARGET_TABLE, 
            properties=write_properties # Make sure you use the bulkCopy properties
        )
    print(f"Successfully wrote data with increased parallelism.")
except Exception as e:
    print(f"An error occurred during write operation: {e}")
    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds")         